In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_D.xpt"
csv_file = DATA_DIR / "BMX_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source for final fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 BMX_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",

    "BMXWT",
    "BMIWT",

    "BMXRECUM",
    "BMIRECUM",

    "BMXHEAD",
    "BMIHEAD",

    "BMXHT",
    "BMIHT",

    "BMXBMI",

    "BMXLEG",
    "BMILEG",

    "BMXCALF",
    "BMICALF",

    "BMXARML",
    "BMIARML",

    "BMXARMC",
    "BMIARMC",

    "BMXWAIST",
    "BMIWAIST",

    "BMXTHICR",
    "BMITHICR",

    "BMXTRI",
    "BMITRI",

    "BMXSUB",
    "BMISUB"
]


if len(df) != 9950:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 27:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "BMX_D columns/order differ from expected.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: BMX_D structure = "
    "9,950 rows x 27 columns."
)


# ============================================================
# 4. DEFINE TRUE BODY-MEASUREMENT VARIABLES
# ============================================================

#
# CRITICAL:
#
# These are MEASUREMENTS.
#
# Do NOT:
# - convert to Int64
# - round()
# - recalculate
# - trim .0
# - change floating-point representation
#

MEASUREMENT_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXCALF",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXTHICR",
    "BMXTRI",
    "BMXSUB"
]


# ============================================================
# 5. DEFINE STATUS / COMMENT CODE FIELDS
# ============================================================

CODE_COLUMNS = [
    "BMDSTATS",
    "BMIWT",
    "BMIRECUM",
    "BMIHEAD",
    "BMIHT",
    "BMILEG",
    "BMICALF",
    "BMIARML",
    "BMIARMC",
    "BMIWAIST",
    "BMITHICR",
    "BMITRI",
    "BMISUB"
]


print(
    "Measurement fields:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "Code/status fields:",
    len(CODE_COLUMNS)
)


# ============================================================
# 6. CHECK FOR XPORT TINY-VALUE ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )

    raise ValueError(
        "STOP: BMX_D unexpectedly contains the "
        "XPORT tiny-value artifact. "
        "Do NOT automatically replace it."
    )


print(
    "PASS: BMX_D contains no tiny-value artifacts."
)


# ============================================================
# 7. TRUE ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\nTrue numeric zeros:", true_zero_total)


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
    )


#
# Current attached BMX_D has zero true numeric zeros.
#

if true_zero_total != 0:

    raise ValueError(
        "Unexpected zero values detected. "
        "Review before proceeding."
    )


print(
    "PASS: Source contains no numeric zero values."
)


# ============================================================
# 8. VERIFY SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN SOURCE CHECK ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31127
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN source values validated."
)


# ============================================================
# 9. VERIFY STATUS / COMMENT VARIABLES ARE WHOLE NUMBERS
# ============================================================

print("\n--- CODE FIELD FRACTION CHECK ---")


fraction_failures = []


for col in CODE_COLUMNS:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        fraction_failures.append(
            col
        )

        print(
            f"Unexpected decimals in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )


if fraction_failures:

    raise ValueError(
        "STOP: Genuine decimal values found "
        "in status/comment fields:\n"
        + str(fraction_failures)
    )


print(
    "PASS: All status/comment fields "
    "contain whole-number codes."
)


# ============================================================
# 10. REPORT GENUINE MEASUREMENT DECIMALS
# ============================================================

print("\n--- BODY MEASUREMENT DECIMAL CHECK ---")


fractional_measurements = []


for col in MEASUREMENT_COLUMNS:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:
        fractional_measurements.append(
            col
        )


    print(
        f"{col:<10}"
        f" valid={int(df[col].notna().sum()):>5,}"
        f" missing={int(df[col].isna().sum()):>5,}"
        f" min={df[col].min()}"
        f" max={df[col].max()}"
        f" fractional={count:,}"
    )


if set(
    fractional_measurements
) != set(
    MEASUREMENT_COLUMNS
):

    raise ValueError(
        "Unexpected measurement decimal pattern."
    )


print(
    "\nPASS: All 13 BMX measurement fields contain "
    "genuine measurement decimals and will remain AS IS."
)


# ============================================================
# 11. CDC BMDSTATS CHECK
# ============================================================

EXPECTED_BMDSTATS = {
    1: 7910,
    2: 196,
    3: 1766,
    4: 78
}


actual_bmdstats = (
    df["BMDSTATS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_bmdstats != EXPECTED_BMDSTATS:

    raise ValueError(
        "BMDSTATS differs from CDC.\n"
        f"Actual: {actual_bmdstats}"
    )


if int(
    df["BMDSTATS"].isna().sum()
) != 0:

    raise ValueError(
        "Unexpected missing BMDSTATS."
    )


print(
    "\nPASS: BMDSTATS matches CDC."
)


# ============================================================
# 12. CDC BODY MEASUREMENT CHECKS
# ============================================================

MEASUREMENT_CHECKS = {

    "BMXWT": {
        "valid": 9847,
        "missing": 103,
        "min": 2.6,
        "max": 371.0
    },

    "BMXRECUM": {
        "valid": 1356,
        "missing": 8594,
        "min": 46.8,
        "max": 113.3
    },

    "BMXHEAD": {
        "valid": 292,
        "missing": 9658,
        "min": 32.6,
        "max": 48.0
    },

    "BMXHT": {
        "valid": 8962,
        "missing": 988,
        "min": 79.8,
        "max": 204.1
    },

    "BMXBMI": {
        "valid": 8949,
        "missing": 1001,
        "min": 11.74,
        "max": 130.21
    },

    "BMXLEG": {
        "valid": 7314,
        "missing": 2636,
        "min": 21.5,
        "max": 53.7
    },

    "BMXCALF": {
        "valid": 7386,
        "missing": 2564,
        "min": 20.2,
        "max": 75.6
    },

    "BMXARML": {
        "valid": 9498,
        "missing": 452,
        "min": 9.5,
        "max": 48.6
    },

    "BMXARMC": {
        "valid": 9494,
        "missing": 456,
        "min": 11.0,
        "max": 62.4
    },

    "BMXWAIST": {
        "valid": 8672,
        "missing": 1278,
        "min": 40.2,
        "max": 175.0
    },

    "BMXTHICR": {
        "valid": 7295,
        "missing": 2655,
        "min": 28.3,
        "max": 93.9
    },

    "BMXTRI": {
        "valid": 8897,
        "missing": 1053,
        "min": 3.0,
        "max": 45.0
    },

    "BMXSUB": {
        "valid": 8110,
        "missing": 1840,
        "min": 2.8,
        "max": 44.0
    }
}


print("\n--- CDC MEASUREMENT CHECKS ---")


for col, spec in MEASUREMENT_CHECKS.items():

    s = df[col]


    if int(
        s.notna().sum()
    ) != spec["valid"]:

        raise ValueError(
            f"{col}: valid count differs."
        )


    if int(
        s.isna().sum()
    ) != spec["missing"]:

        raise ValueError(
            f"{col}: missing count differs."
        )


    if not np.isclose(
        s.min(),
        spec["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: minimum differs."
        )


    if not np.isclose(
        s.max(),
        spec["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col}: maximum differs."
        )


    print(
        f"PASS: {col:<10}"
        f" range={s.min()}-{s.max()}"
        f" valid={int(s.notna().sum()):,}"
        f" missing={int(s.isna().sum()):,}"
    )


# ============================================================
# 13. CDC COMMENT-CODE CHECKS
# ============================================================

EXPECTED_COMMENT_COUNTS = {

    "BMIWT": {
        1: 29,
        3: 282,
        4: 15
    },

    "BMIRECUM": {
        1: 38,
        3: 1
    },

    "BMIHEAD": {},

    "BMIHT": {
        1: 62,
        3: 258
    },

    "BMILEG": {
        1: 325
    },

    "BMICALF": {
        1: 258
    },

    "BMIARML": {
        1: 291
    },

    "BMIARMC": {
        1: 297
    },

    "BMIWAIST": {
        1: 346
    },

    "BMITHICR": {
        1: 350
    },

    "BMITRI": {
        1: 712,
        2: 183
    },

    "BMISUB": {
        1: 1575,
        2: 108
    }
}


print("\n--- CDC COMMENT CODE CHECKS ---")


for col, expected in EXPECTED_COMMENT_COUNTS.items():

    actual = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    if actual != expected:

        raise ValueError(
            f"{col} differs from CDC.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 14. CLEAN SEQN
# ============================================================

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\nSEQN converted:")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 15. FORMAT ONLY STATUS / COMMENT CODES
# ============================================================

#
# Example:
#
# BMDSTATS
# 1.0 -> 1
#
# BMIWT
# 3.0 -> 3
#
# But BMX measurements remain floats.
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: SEQN and status/comment fields "
    "formatted without unnecessary .0."
)


# ============================================================
# 16. BUILD EXPECTED SOURCE
# ============================================================

#
# BMX_D requires NO value correction.
#
# Expected numeric source is therefore exactly
# the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 17. EXACT VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if (
        col == "SEQN"
        or
        col in CODE_COLUMNS
    ):

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 27 BMX_D variables retain "
    "their original numeric values."
)


# ============================================================
# 18. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO recalculation
# NO decimal trimming
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 19. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found in:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 27 CSV columns numerically "
    "match original BMX_D XPT."
)


# ============================================================
# 21. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            csv_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 22. READ RAW CSV TEXT
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 23. SEQN / CODE .0 FORMATTING CHECK
# ============================================================

bad_dot_zero = 0
bad_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in ["SEQN"] + CODE_COLUMNS:

        value = row[col]

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            bad_dot_zero += 1

            if len(
                bad_examples
            ) < 20:

                bad_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nSEQN/code values ending '.0':",
    bad_dot_zero
)


if bad_dot_zero != 0:

    print(
        "Examples:",
        bad_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in "
        "SEQN/status/comment fields."
    )


print(
    "PASS: No unnecessary .0 in "
    "SEQN/status/comment fields."
)


# ============================================================
# 24. RAW BMX MEASUREMENT FIDELITY
# ============================================================

#
# This is the strict formatting check.
#
# Compare raw CSV token against:
#
# str(float(original XPT value))
#
# Therefore:
#
# 48.0 stays 48.0
# 175.0 stays 175.0
# 23.5 stays 23.5
#
# and any longer floating-point representation
# remains exactly as pandas read it from XPT.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(
                source_value
            )
            else str(
                float(
                    source_value
                )
            )
        )


        actual_text = row[col]


        if actual_text != expected_text:

            measurement_mismatches[
                col
            ] += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


bad_measurements = {
    col: count
    for col, count
    in measurement_mismatches.items()
    if count > 0
}


print(
    "BMX measurement raw-text mismatches:",
    bad_measurements
)


if bad_measurements:

    print(
        "Examples:",
        measurement_examples
    )

    raise ValueError(
        "Body-measurement formatting changed "
        "during CSV export."
    )


print(
    "PASS: All 13 BMX measurement variables "
    "appear AS IS in CSV."
)


# ============================================================
# 25. LEGITIMATE MEASUREMENT .0 REPORT
# ============================================================

measurement_dot_zero = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for row in raw_rows:

    for col in MEASUREMENT_COLUMNS:

        value = row[col]

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            measurement_dot_zero[
                col
            ] += 1


print(
    "\nLegitimate measurement values ending '.0':"
)


for col, count in measurement_dot_zero.items():

    if count > 0:

        print(
            f"{col:<10}: {count:,}"
        )


print(
    "\nIMPORTANT: These are measurement values "
    "and should NOT be changed."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower_value = (
            value.lower()
        )


        if (
            "e-" in lower_value
            or
            "e+" in lower_value
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("BMX_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Measurement fields:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "Status/comment fields:",
    len(CODE_COLUMNS)
)

print(
    "Tiny-value corrections:",
    0
)

print(
    "XPT -> CSV differences:",
    0
)

print(
    "Missing-position differences:",
    0
)

print(
    "Expected SEQN/code .0:",
    0
)

NHANES 2005-2006 BMX_D XPT -> CSV
Rows: 9,950
Columns: 27
PASS: BMX_D structure = 9,950 rows x 27 columns.
Measurement fields: 13
Code/status fields: 13

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: BMX_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Source contains no numeric zero values.

--- SEQN SOURCE CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31127.0
Maximum: 41474.0
PASS: SEQN source values validated.

--- CODE FIELD FRACTION CHECK ---
PASS: All status/comment fields contain whole-number codes.

--- BODY MEASUREMENT DECIMAL CHECK ---
BMXWT      valid=9,847 missing=  103 min=2.6 max=371.0 fractional=8,839
BMXRECUM   valid=1,356 missing=8,594 min=46.8 max=113.3 fractional=1,202
BMXHEAD    valid=  292 missing=9,658 min=32.6 max=48.0 fractional=263
BMXHT      valid=8,962 missing=  988 min=79.8 max=204.1 fractional=8,094
BMXBMI     valid=8,949 missing=1,001 min=11.74 max=130.21 fractional=8,841
BMXLEG     valid=7,314 missing=2,636 min=21.5 max=53.7 fr